# Persistence Model and Storage Boundary (Slice 4A)

Feature 1–3 turn raw SEC filings into canonical facts, deterministic metrics, and an Economic Value Lens — but every run so far is **in-memory only**. This notebook adds OwnerLens's first persistence layer: a small local store that keeps company identity, source-snapshot metadata, canonical facts, derived metrics, Feature 2 analysis outputs, and coverage states so they can be reproduced and queried later without re-fetching from SEC.

Persistence stays **strictly downstream** of computation. It stores outputs; it never normalizes, calculates, or classifies.

We answer eight questions:

1. Why does OwnerLens need persistence?
2. Facts vs metrics vs analysis outputs
3. Persist ADBE, Visa, and Costco
4. Why provenance survives persistence
5. Why absence / unsupported / insufficient are **not** SQL `NULL`
6. Historical snapshots versus one mutable latest row
7. Query several useful owner/investor views
8. How this storage boundary maps to Azure Blob + PostgreSQL next

> Requires `OWNER_LENS_SEC_USER_AGENT` for the live SEC calls below.

In [2]:
import json
import os
import sqlite3
import tempfile
from pathlib import Path

from owner_lens import (
    FilesystemRawSnapshotStore,
    SecClient,
    SourceSnapshotRecord,
    SqliteStore,
    analysis_result_records,
    capital_allocation_from_facts,
    capital_efficiency_from_facts,
    company_coverage,
    company_record,
    compounding_views_from_facts,
    compute_content_hash,
    coverage_result_records,
    derived_metric_records,
    economic_value_from_facts,
    economic_value_summary_from_facts,
    owner_economics_from_facts,
    reported_fact_records,
)

USER_AGENT = os.environ.get('OWNER_LENS_SEC_USER_AGENT', 'OwnerLens admin@example.com')
TICKERS = ['ADBE', 'V', 'COST']

with SecClient(USER_AGENT) as client:
    results = {t: client.retrieve_company_facts(t) for t in TICKERS}

facts = {t: r.raw_facts for t, r in results.items()}
identities = {t: r.identity for t, r in results.items()}
{t: identities[t].cik for t in TICKERS}

{'ADBE': '0000796343', 'V': '0001403161', 'COST': '0000909832'}

## 1. Why does OwnerLens need persistence?

So far each analysis is recomputed from a live SEC fetch and then discarded. Persistence lets OwnerLens:

* **reproduce** a prior result in a fresh session without re-fetching,
* **query** history (for example five years of FCF/share) and provenance,
* **retain** what it knew after a given filing, and
* prepare for batch ingestion of many companies later.

The store keeps four kinds of output — reported **facts**, derived **metrics**, **analysis** results, and **coverage** — each tied to the source snapshot that produced it.

## 2. Facts vs metrics vs analysis outputs

These are deliberately separate tables, not one wide row:

* **Reported facts** are canonical observations with full SEC provenance (e.g. `revenue` from the concept `RevenueFromContractWithCustomerExcludingAssessedTax`).
* **Derived metrics** are deterministic calculations (FCF, ROIC, FCF/share). Money and share counts are exact `INTEGER`; ratios are `REAL` (floating point — approximate, compared with tolerances).
* **Analysis results** are Feature 2 classifications with ordered drivers.

The helper below converts existing domain outputs into records via pure adapters, then saves them through the explicit store API — no financial function ever writes to storage itself.

In [3]:
FETCHED_AT = '2026-09-04T00:00:00'


def persist(store: SqliteStore, ticker: str) -> SourceSnapshotRecord:
    """Compute the full pipeline for one company and persist every layer."""
    identity = identities[ticker]
    payload = facts[ticker]
    cik = identity.cik

    owner_rows = owner_economics_from_facts(payload, ticker=ticker)
    capital_rows = capital_efficiency_from_facts(payload, ticker=ticker)
    snapshots = economic_value_from_facts(payload, ticker=ticker)
    views = compounding_views_from_facts(payload, ticker=ticker)
    allocation = capital_allocation_from_facts(payload, ticker=ticker)
    summary = economic_value_summary_from_facts(payload, ticker=ticker)
    coverage = company_coverage(payload, ticker=ticker)

    raw_bytes = json.dumps(payload, sort_keys=True).encode()
    snapshot = SourceSnapshotRecord(
        cik=cik,
        source_provider='sec',
        source_type='company_facts',
        fetched_at=FETCHED_AT,
        source_uri=f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json',
        content_hash=compute_content_hash(raw_bytes),
    )

    store.save_company(company_record(identity))
    store.save_source_snapshot(snapshot, raw_payload=raw_bytes)
    store.save_reported_facts(
        reported_fact_records(cik, owner_rows, capital_rows), snapshot=snapshot
    )
    store.save_derived_metrics(
        derived_metric_records(cik, owner_rows, capital_rows, computed_at=FETCHED_AT),
        snapshot=snapshot,
    )
    for result in analysis_result_records(
        cik,
        annual_snapshots=snapshots,
        compounding_views=views,
        capital_allocation_rows=allocation,
        summary=summary,
        computed_at=FETCHED_AT,
    ):
        store.save_analysis_result(result, snapshot=snapshot)
    store.save_coverage(coverage_result_records(cik, coverage), snapshot=snapshot)
    return snapshot

## 3. Persist ADBE, Visa, and Costco

We initialize a fresh local SQLite store (with a filesystem raw-payload store beside it) and persist all three companies. SQLite needs no external service, keeps relational semantics close to a future PostgreSQL, and is easy to validate locally.

In [4]:
workdir = Path(tempfile.mkdtemp(prefix='owner_lens_'))
db_path = workdir / 'owner_lens.db'
raw_store = FilesystemRawSnapshotStore(workdir / 'raw')

store = SqliteStore(db_path, raw_store=raw_store)
store.initialize()

snapshots = {t: persist(store, t) for t in TICKERS}
{t: snapshots[t].content_hash[:12] for t in TICKERS}

{'ADBE': '11c445b544ad', 'V': '783a56fbf222', 'COST': '1c812fb0237b'}

## 4. Why provenance survives persistence

A stored fact keeps the SEC concept, form, filing date, and accession that produced it, and the canonical metric name stays distinct from the source concept. Costco's FY2025 revenue is a good example — the canonical `revenue` came from a contract-revenue concept, not a tag literally named "revenue".

In [5]:
cost_cik = identities['COST'].cik
revenue = store.get_fact_provenance(cost_cik, 'revenue', 2025)
print('canonical metric:', revenue.canonical_metric)
print('source concept:  ', revenue.concept)
print('value (USD):     ', revenue.value)
print('form / filed:    ', revenue.form, '/', revenue.filed)
print('accession:       ', revenue.accession)

canonical metric: revenue
source concept:   RevenueFromContractWithCustomerExcludingAssessedTax
value (USD):      275235000000
form / filed:     10-K / 2025-10-08
accession:        0000909832-25-000101


## 5. Why absence / unsupported / insufficient are not SQL `NULL`

OwnerLens preserves four distinct meanings that a naive schema would flatten into `NULL`:

* **`UNSUPPORTED`** — the concept does not exist for this filer (Visa's weighted-average diluted shares).
* **`STRUCTURALLY_ABSENT`** — a policy gap (e.g. no dividend program).
* **`INSUFFICIENT_DATA`** — a layer could not be classified because an input was missing.
* **reported `0`** — a genuine observation, stored as a fact with `value == 0`.

Coverage states are stored as explicit strings in a dedicated table; an unavailable metric is a **missing fact row** plus a coverage reason, never a `NULL` value pretending to be a number.

In [6]:
visa_cik = identities['V'].cik
coverage = store.get_company_coverage(visa_cik)

inputs = {c.subject: c.state for c in coverage if c.subject_kind == 'input'}
layer_states = sorted({c.state for c in coverage if c.subject_kind == 'layer'})
print('Visa diluted_shares coverage:', inputs['diluted_shares'])
print('Visa layer states:          ', layer_states)

# A reported zero stays a real observation, distinct from absence.
adbe_cik = identities['ADBE'].cik
zero = store.get_fact_provenance(adbe_cik, 'current_debt', 2023)
print('ADBE current_debt 2023 value:', None if zero is None else zero.value)

Visa diluted_shares coverage: UNSUPPORTED
Visa layer states:           ['AVAILABLE', 'INSUFFICIENT_DATA', 'PARTIAL']
ADBE current_debt 2023 value: 0


## 6. Historical snapshots versus one mutable latest row

Every fact, metric, analysis, and coverage row references the **source snapshot** that produced it. Re-processing the identical payload (same content hash) is idempotent — it adds no rows — while a genuinely new filing becomes a new snapshot that coexists with the old one. This is what lets OwnerLens later answer "what did we know after this filing?" and distinguish a value that changed because the *data* changed from one that changed because the *calculation rules* changed.

In [7]:
def count(table: str) -> int:
    with sqlite3.connect(db_path) as conn:
        return int(conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0])


before = count('reported_facts')
persist(store, 'ADBE')  # identical repeat: same content hash
after = count('reported_facts')
print('reported_facts rows before / after identical repeat:', before, '/', after)

reported_facts rows before / after identical repeat: 170 / 170


## 7. Query several useful owner/investor views

With the data persisted, common owner questions become simple reads: a multi-year metric history, the latest company summary, and a single primitive compared across companies (preparation for future scoring, without building a comparison feature here).

In [8]:
# ADBE FCF/share history (latest five fiscal years available).
history = store.get_metric_history(adbe_cik, 'fcf_per_share', limit=5)
print('ADBE FCF/share history:')
for record in history:
    print(f'  FY{record.fiscal_year}: {record.value_real:.2f}')

# Latest ADBE Economic Value Summary.
summary = store.get_latest_summary(adbe_cik)
print('\nADBE latest summary:', summary.classification)
print('  drivers:', [d.code for d in summary.drivers])

# Latest ROIC across ADBE / V / COST.
latest_roic = store.get_latest_metric_across_companies(
    'roic', [adbe_cik, visa_cik, cost_cik]
)
print('\nLatest ROIC:')
for ticker in TICKERS:
    record = latest_roic.get(identities[ticker].cik)
    if record is not None:
        print(f'  {ticker}: {record.value_real:.1%} (FY{record.fiscal_year})')

ADBE FCF/share history:
  FY2021: 14.31
  FY2022: 15.71
  FY2023: 15.12
  FY2024: 17.51
  FY2025: 23.07

ADBE latest summary: IMPROVING
  drivers: ['PER_SHARE_CASH_FLOW_COMPOUNDING', 'HIGH_ROIC', 'ROIC_IMPROVING', 'SHARE_COUNT_SHRINKING', 'EFFECTIVE_BUYBACKS', 'OWNER_FRIENDLY_CAPITAL_ALLOCATION', 'HIGH_SBC_BURDEN', 'CAPITAL_RETURNS_EXCEED_FCF', 'DECLINING_NET_CASH_CUSHION', 'MARGINS_CONTRACTING']

Latest ROIC:
  ADBE: 61.6% (FY2025)
  V: 42.4% (FY2025)
  COST: 40.9% (FY2025)


## 8. How this storage boundary maps to Azure Blob + PostgreSQL next

Slice 4A is deliberately local-first, but the shape is Azure-ready:

* **Raw payloads** are already referenced by content hash through a small `RawSnapshotStore` protocol. The local `FilesystemRawSnapshotStore` can be swapped for a `BlobRawSnapshotStore` in Slice 4B with **no relational schema change**.
* **The relational model** uses portable SQL and standard types, so the same tables move to **Azure PostgreSQL** — the driver, not the design, changes.
* **Errors** are a distinct `PersistenceError` hierarchy, so cloud connection or retry concerns stay separate from financial semantics.

What we deliberately did **not** build: Azure SDKs, provisioning, batch/large-universe ingestion, migrations, an API service, or any scoring/comparison feature. Persistence remains downstream of trustworthy computation — it stores what OwnerLens already knows, faithfully.